# Проект: Анализ CSV-датасета (Kaggle Raw Data)

**Команда:** №___  
**Участники:** ФИО 1 (группа ___), ФИО 2 (группа ___)

Источник данных: https://www.kaggle.com/datasets/sanarpit/raw-data

## 1. Описание данных

В этом проекте анализируется CSV-датасет из Kaggle. Наблюдение — одна строка таблицы (одна запись объекта из предметной области).  
Ниже код, который загружает файл, показывает размерность данных и типы признаков.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

DATA_PATH = 'data/raw-data.csv'  # Скачайте CSV из Kaggle и положите сюда

df = pd.read_csv(DATA_PATH)
print(f'Размер датасета: {df.shape[0]} наблюдений и {df.shape[1]} признаков')
display(df.head())

In [ ]:
# Таблица признаков: название, тип данных, логический тип признака
feature_table = pd.DataFrame({
    'Признак': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
})

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

feature_table['Тип признака'] = feature_table['Признак'].apply(
    lambda x: 'количественный' if x in num_cols else 'категориальный'
)
feature_table['Описание (заполните предметно)'] = 'Описание по смыслу признака'

display(feature_table)

### Интерпретация описательных статистик
- Количественные признаки анализируются по среднему, медиане, квартилям и разбросу.
- Категориальные признаки анализируются по моде и частотам значений.
- Если среднее сильно отличается от медианы, распределение может быть асимметричным.
- Большой размах и высокий IQR могут указывать на потенциальные выбросы.

In [ ]:
# Описательные статистики
if num_cols:
    display(df[num_cols].describe().T)
if cat_cols:
    display(df[cat_cols].describe(include='all').T)

### Уравнение линейной регрессии

Для простой линейной регрессии (один предиктор $x$ и целевая переменная $y$):

$$
\hat{y} = b_0 + b_1 x,
$$

где

$$
b_1 = rac{\sum (x_i-ar{x})(y_i-ar{y})}{\sum (x_i-ar{x})^2}, \qquad
b_0 = ar{y} - b_1ar{x}.
$$

In [ ]:
# Расчет коэффициентов простой линейной регрессии вручную
if len(num_cols) >= 2:
    x_col, y_col = num_cols[0], num_cols[1]
    x = df[x_col].dropna()
    y = df.loc[x.index, y_col].dropna()
    common_idx = x.index.intersection(y.index)
    x = df.loc[common_idx, x_col]
    y = df.loc[common_idx, y_col]

    x_mean, y_mean = x.mean(), y.mean()
    b1 = ((x - x_mean) * (y - y_mean)).sum() / ((x - x_mean) ** 2).sum()
    b0 = y_mean - b1 * x_mean
    print(f'Выбраны признаки: x={x_col}, y={y_col}')
    print(f'Уравнение: y_hat = {b0:.4f} + {b1:.4f} * x')
else:
    print('Недостаточно количественных признаков для простой регрессии.')

## 2. Пропуски
Проверяем пропуски. Числовые заменяем медианой, категориальные — модой.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

df_filled = df.copy()
for c in num_cols:
    if df_filled[c].isna().any():
        df_filled[c] = df_filled[c].fillna(df_filled[c].median())
for c in cat_cols:
    if df_filled[c].isna().any():
        df_filled[c] = df_filled[c].fillna(df_filled[c].mode().iloc[0])

print('Пропуски после замены:', int(df_filled.isna().sum().sum()))

## 3. Выбросы
1) Удаление выбросов по целевому признаку методом $N$ стандартных отклонений.  
2) Ящик с усами по количественному признаку (не целевому).  
3) Удаление выбросов методом $1.5 \cdot IQR$ для выбранного признака.

In [ ]:
if num_cols:
    target_col = num_cols[-1]
    N = 3
    mu = df_filled[target_col].mean()
    sigma = df_filled[target_col].std(ddof=0)
    mask_sigma = (df_filled[target_col] >= mu - N*sigma) & (df_filled[target_col] <= mu + N*sigma)
    df_no_outliers_sigma = df_filled.loc[mask_sigma].copy()
    print(f'Целевой признак: {target_col}')
    print('Размер до:', df_filled.shape, 'после N-sigma:', df_no_outliers_sigma.shape)
else:
    df_no_outliers_sigma = df_filled.copy()
    print('Нет числовых признаков для удаления выбросов методом N-sigma')

In [ ]:
if len(num_cols) >= 2:
    box_col = num_cols[0]
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=df_no_outliers_sigma[box_col])
    plt.title(f'Ящик с усами: {box_col}')
    plt.show()

    q1 = df_no_outliers_sigma[box_col].quantile(0.25)
    q3 = df_no_outliers_sigma[box_col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df_clean = df_no_outliers_sigma[(df_no_outliers_sigma[box_col] >= low) & (df_no_outliers_sigma[box_col] <= high)].copy()
    print('Размер после 1.5*IQR:', df_clean.shape)
else:
    df_clean = df_no_outliers_sigma.copy()
    print('Недостаточно числовых признаков для boxplot/IQR')

## 4. Фильтрация и сортировка

In [ ]:
# По одному условию: сортировка по возрастанию и убыванию
if num_cols:
    c = num_cols[0]
    thr = df_clean[c].median()
    one_cond = df_clean[df_clean[c] > thr]
    display(one_cond.sort_values(by=c, ascending=True).head(10))
    display(one_cond.sort_values(by=c, ascending=False).head(10))

In [ ]:
# По нескольким условиям: сортировка по возрастанию и убыванию
if num_cols and cat_cols:
    c_num = num_cols[0]
    c_cat = cat_cols[0]
    m = df_clean[c_num].median()
    top_cat = df_clean[c_cat].mode().iloc[0]

    multi_cond = df_clean[(df_clean[c_num] >= m) & (df_clean[c_cat] == top_cat)]
    display(multi_cond.sort_values(by=[c_num], ascending=True).head(10))
    display(multi_cond.sort_values(by=[c_num], ascending=False).head(10))

**Выводы по фильтрации и сортировке**
- Сортировка по возрастанию помогает увидеть минимальные значения в отфильтрованной группе.
- Сортировка по убыванию показывает экстремально большие значения.
- При нескольких условиях выборка более «целевая», но размер подтаблицы уменьшается.

## 5. Новые признаки
1) Бинарный признак через анонимную функцию.  
2) Новый признак с использованием `in/.count()/.len()` в анонимной функции.  
3) Новый признак с использованием именованной функции.

In [ ]:
df_feat = df_clean.copy()

# 1) Бинарный признак
if num_cols:
    base_num = num_cols[0]
    med = df_feat[base_num].median()
    df_feat['is_high_' + base_num] = df_feat[base_num].apply(lambda x: 1 if x >= med else 0)

# 2) Признак на базе строкового признака
if cat_cols:
    base_cat = cat_cols[0]
    df_feat['len_' + base_cat] = df_feat[base_cat].astype(str).apply(lambda s: len(s))
    df_feat['contains_space_' + base_cat] = df_feat[base_cat].astype(str).apply(lambda s: 1 if ' ' in s else 0)

# 3) Именованная функция
def group_size_label(n):
    if n < 5:
        return 'short'
    elif n < 10:
        return 'medium'
    return 'long'

if cat_cols:
    df_feat['name_length_group'] = df_feat['len_' + base_cat].apply(group_size_label)

display(df_feat.head())

**Выводы по новым признакам**
- Бинаризация упрощает дальнейшую сегментацию и сравнение групп.
- Длина строки и наличие пробела могут быть полезны для текстовых категориальных признаков.
- Группировка длины в категории повышает интерпретируемость.

## 6. Частотные таблицы

In [ ]:
if cat_cols:
    c = cat_cols[0]
    freq = df_feat[c].value_counts()
    display(freq.sort_values(ascending=False).to_frame('count_desc_values'))
    display(freq.sort_values(ascending=True).to_frame('count_asc_values'))
    display(freq.sort_index(ascending=True).to_frame('count_asc_index'))
    display(freq.sort_index(ascending=False).to_frame('count_desc_index'))

**Выводы по частотным таблицам**
- Таблица по убыванию значений быстро выявляет доминирующие категории.
- Таблица по возрастанию значений помогает увидеть редкие категории.
- Сортировка по индексу удобна для алфавитного/логического порядка категорий.

## 7. Сводные таблицы

In [ ]:
if num_cols and cat_cols:
    c_cat = cat_cols[0]
    c_num = num_cols[0]

    # 1) один столбец группировки, один агрегируемый, один метод
    p1 = pd.pivot_table(df_feat, index=c_cat, values=c_num, aggfunc='mean')
    display(p1)

    # 2) несколько столбцов группировки, один агрегируемый, один метод
    if len(cat_cols) >= 2:
        p2 = pd.pivot_table(df_feat, index=[cat_cols[0], cat_cols[1]], values=c_num, aggfunc='mean')
        display(p2)

    # 3) один столбец группировки, несколько агрегируемых, один метод
    use_nums = num_cols[:min(3, len(num_cols))]
    p3 = pd.pivot_table(df_feat, index=c_cat, values=use_nums, aggfunc='mean')
    display(p3)

    # 4) один столбец группировки, один агрегируемый, несколько методов
    p4 = pd.pivot_table(df_feat, index=c_cat, values=c_num, aggfunc=['mean', 'median', 'min', 'max'])
    display(p4)

**Выводы по сводным таблицам**
- Сводные таблицы позволяют компактно сравнивать группы по статистикам.
- Добавление нескольких уровней группировки уточняет различия между подгруппами.
- Использование нескольких агрегирующих функций дает более полную картину распределения.

## 8. Визуализация

In [ ]:
if num_cols:
    q = num_cols[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df_feat[q], kde=True, bins=30)
    plt.title(f'Гистограмма количественного признака: {q}')
    plt.show()

**Интерпретация гистограммы**
- Форма распределения (симметрия/асимметрия) показывает структуру данных.
- Наличие длинных хвостов может указывать на редкие экстремальные наблюдения.

In [ ]:
if cat_cols:
    c = cat_cols[0]
    vc = df_feat[c].value_counts().head(10)

    plt.figure(figsize=(10,4))
    sns.barplot(x=vc.index, y=vc.values)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Столбчатая диаграмма категориального признака: {c} (top-10)')
    plt.ylabel('Количество')
    plt.show()

    plt.figure(figsize=(6,6))
    vc.plot(kind='pie', autopct='%1.1f%%')
    plt.title(f'Круговая диаграмма категориального признака: {c} (top-10)')
    plt.ylabel('')
    plt.show()

**Интерпретация столбчатой и круговой диаграмм**
- Видно, какие категории наиболее распространены.
- Если одна категория существенно больше других, распределение категорий несбалансировано.

## 9. Итог

В проекте выполнены:
- описание данных и признаков;
- обработка пропусков;
- удаление выбросов двумя подходами;
- фильтрация и сортировка;
- создание новых признаков;
- частотные и сводные таблицы;
- визуализации и интерпретации.

> Перед сдачей: заполните предметные описания признаков и блок с участниками команды.